In [ ]:
# Imports i konfiguracja
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize
import pathlib

SEED = 42
np.random.seed(SEED)

# Ścieżki
DATA_DIR = pathlib.Path("../data")
REPORTS_DIR = pathlib.Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Klasy
CLASS_NAMES = ["non-toxic", "toxic", "severely-toxic"]
N_CLASSES = 3

## 1. Load Data

In [ ]:
# Wczytaj dane
df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_val = pd.read_csv(DATA_DIR / "df_val.csv")
df_test_sample = pd.read_csv(DATA_DIR / "df_test_sample.csv")

# Wczytaj indeksy sample
with open(DATA_DIR / "sample_indices.txt", "r") as f:
    sample_indices = eval(f.read())

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test Sample: {len(df_test_sample)}")
print(f"\nTest sample class distribution:")
print(df_test_sample['toxic_level'].value_counts().sort_index())

In [ ]:
# Przygotuj dane
X_train = df_train['comment_text'].fillna('')
y_train = df_train['toxic_level'].values

X_val = df_val['comment_text'].fillna('')
y_val = df_val['toxic_level'].values

X_test = df_test_sample['comment_text'].fillna('')
y_test = df_test_sample['toxic_level'].values

print(f"X_train: {len(X_train)}, X_test: {len(X_test)}")

## 2. Baseline: TF-IDF + Logistic Regression (3-class)

In [ ]:
# TF-IDF + LogReg pipeline
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
)

clf_baseline = LogisticRegression(
    solver="lbfgs",
    class_weight="balanced",
    C=2.0,
    max_iter=1000,
    random_state=SEED,
    multi_class="multinomial",
)

pipe_baseline = Pipeline([("tfidf", tfidf), ("clf", clf_baseline)])
pipe_baseline.fit(X_train, y_train)
print("Baseline model trained")

In [ ]:
# Predykcje baseline
y_pred_baseline = pipe_baseline.predict(X_test)
y_prob_baseline = pipe_baseline.predict_proba(X_test)

print("Baseline predictions done")
print(f"Unique predictions: {np.unique(y_pred_baseline, return_counts=True)}")

## 3. XGBoost (from advanced-model)

In [ ]:
import tensorflow as tf
import xgboost as xgb

# TextVectorization layer (same as advanced-model.ipynb)
max_features = 100000
sequence_length = 1024

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=max_features,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Adapt on training data
vectorize_layer.adapt(X_train.values)
print(f"Vocabulary size: {len(vectorize_layer.get_vocabulary())}")

In [ ]:
# Vectorize data
def vectorize_texts(texts):
    return vectorize_layer(texts).numpy()

X_train_vec = vectorize_texts(X_train.values)
X_test_vec = vectorize_texts(X_test.values)

print(f"X_train_vec shape: {X_train_vec.shape}")
print(f"X_test_vec shape: {X_test_vec.shape}")

In [ ]:
# Train XGBoost
dtrain = xgb.DMatrix(X_train_vec, label=y_train)
dtest = xgb.DMatrix(X_test_vec, label=y_test)

param = {
    'max_depth': 4,
    'eta': 1,
    'objective': 'multi:softprob',  # Use softprob for probabilities
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'nthread': 4,
    'seed': SEED
}

num_round = 25
bst = xgb.train(param, dtrain, num_round, verbose_eval=False)
print("XGBoost model trained")

In [ ]:
# XGBoost predictions
y_prob_xgb = bst.predict(dtest)
y_pred_xgb = np.argmax(y_prob_xgb, axis=1)

print("XGBoost predictions done")
print(f"Unique predictions: {np.unique(y_pred_xgb, return_counts=True)}")

## 4. BERT (from saved predictions)

In [ ]:
# Load BERT predictions (128 tokens version)
with open("BERTpred128.txt", "r") as f:
    bert_preds_full = eval(f.read())

# Extract predictions for our sample indices
y_pred_bert = np.array([bert_preds_full[i] for i in sample_indices])

print(f"BERT predictions extracted: {len(y_pred_bert)}")
print(f"Unique predictions: {np.unique(y_pred_bert, return_counts=True)}")

## 5. LLM: Groq API (Llama 3.1 70B)

**Uwaga**: Wymaga klucza API Groq. Ustaw zmienną środowiskową `GROQ_API_KEY`.

Darmowy tier: 6000 requestów/dzień, 30 req/min.

In [ ]:
# LLM Configuration
LLM_MODEL = "llama-3.1-70b-versatile"  # or "mixtral-8x7b-32768"
LLM_PREDICTIONS_FILE = DATA_DIR / "llm_predictions.json"

# Check if we have cached predictions
if LLM_PREDICTIONS_FILE.exists():
    with open(LLM_PREDICTIONS_FILE, "r") as f:
        llm_cache = json.load(f)
    print(f"Loaded {len(llm_cache)} cached LLM predictions")
else:
    llm_cache = {}
    print("No cached predictions found")

In [ ]:
import time
import re

# Try to import groq
try:
    from groq import Groq
    GROQ_AVAILABLE = True
except ImportError:
    print("Groq not installed. Run: pip install groq")
    GROQ_AVAILABLE = False

def classify_with_llm(client, text, max_retries=3):
    """Classify text using LLM with temperature=0."""
    prompt = f"""Classify the following comment as one of: 0 (non-toxic), 1 (toxic), or 2 (severely toxic).

Rules:
- 0 = non-toxic: Normal, respectful discussion
- 1 = toxic: Rude, disrespectful, or mildly offensive
- 2 = severely toxic: Hate speech, threats, extreme profanity

Respond with ONLY a single digit: 0, 1, or 2.

Comment: {text[:1500]}

Classification:"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=10,
            )
            answer = response.choices[0].message.content.strip()
            
            # Extract digit
            match = re.search(r'[012]', answer)
            if match:
                return int(match.group())
            else:
                print(f"Invalid response: {answer}")
                return 0  # Default to non-toxic
                
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    
    return 0  # Default on failure

In [ ]:
# Run LLM inference (with rate limiting and caching)
if GROQ_AVAILABLE and os.environ.get("GROQ_API_KEY"):
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
    
    y_pred_llm = []
    
    for i, (idx, text) in enumerate(zip(df_test_sample['original_idx'], X_test)):
        cache_key = str(idx)
        
        if cache_key in llm_cache:
            pred = llm_cache[cache_key]
        else:
            pred = classify_with_llm(client, text)
            llm_cache[cache_key] = pred
            
            # Rate limiting: 30 req/min
            if (i + 1) % 25 == 0:
                time.sleep(60)  # Wait 1 minute every 25 requests
                
            # Save cache periodically
            if (i + 1) % 50 == 0:
                with open(LLM_PREDICTIONS_FILE, "w") as f:
                    json.dump(llm_cache, f)
                print(f"Processed {i+1}/{len(X_test)} samples")
        
        y_pred_llm.append(pred)
    
    # Final save
    with open(LLM_PREDICTIONS_FILE, "w") as f:
        json.dump(llm_cache, f)
    
    y_pred_llm = np.array(y_pred_llm)
    print(f"\nLLM predictions done: {len(y_pred_llm)}")
    print(f"Unique predictions: {np.unique(y_pred_llm, return_counts=True)}")
    
else:
    print("Groq API not available. Set GROQ_API_KEY environment variable.")
    print("To get free API key: https://console.groq.com/keys")
    y_pred_llm = None

## 6. Metrics Calculation

In [ ]:
def calculate_metrics(y_true, y_pred, y_prob=None, name="Model"):
    """Calculate all required metrics."""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    
    # ROC-AUC (requires probabilities)
    if y_prob is not None:
        y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
        try:
            auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
        except:
            auc = None
    else:
        auc = None
    
    return {
        "model": name,
        "accuracy": acc,
        "macro_f1": f1,
        "roc_auc": auc,
        "predictions": y_pred
    }

In [ ]:
# Calculate metrics for all models
results = []

# Baseline
results.append(calculate_metrics(y_test, y_pred_baseline, y_prob_baseline, "TF-IDF + LogReg"))

# XGBoost
results.append(calculate_metrics(y_test, y_pred_xgb, y_prob_xgb, "XGBoost"))

# BERT
results.append(calculate_metrics(y_test, y_pred_bert, None, "BERT (128)"))

# LLM
if y_pred_llm is not None:
    results.append(calculate_metrics(y_test, y_pred_llm, None, f"LLM ({LLM_MODEL})"))

print("Metrics calculated for all models")

## 7. Comparison Table

In [ ]:
# Create comparison DataFrame
df_results = pd.DataFrame([{
    "Model": r["model"],
    "Accuracy": f"{r['accuracy']:.4f}",
    "Macro-F1 ⭐": f"{r['macro_f1']:.4f}",
    "ROC-AUC": f"{r['roc_auc']:.4f}" if r['roc_auc'] else "N/A"
} for r in results])

# Sort by Macro-F1 (main metric)
df_results_sorted = df_results.copy()
df_results_sorted["_f1_numeric"] = [float(r["macro_f1"]) for r in results]
df_results_sorted = df_results_sorted.sort_values("_f1_numeric", ascending=False).drop(columns=["_f1_numeric"])

print("\n" + "="*60)
print("           MODEL COMPARISON (sorted by Macro-F1)")
print("="*60)
display(df_results_sorted.reset_index(drop=True))

In [ ]:
# Visualization: Bar chart
fig, ax = plt.subplots(figsize=(10, 6))

models = [r["model"] for r in results]
f1_scores = [r["macro_f1"] for r in results]
accuracies = [r["accuracy"] for r in results]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, f1_scores, width, label='Macro-F1', color='steelblue')
bars2 = ax.bar(x + width/2, accuracies, width, label='Accuracy', color='coral')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Hate Speech Detection')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))

if n_models == 1:
    axes = [axes]

for ax, r in zip(axes, results):
    cm = confusion_matrix(y_test, r["predictions"], normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format=".2f")
    ax.set_title(f"{r['model']}\nMacro-F1: {r['macro_f1']:.4f}")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Error Analysis (10 cases)

In [ ]:
# Find misclassified examples
errors = []

for i, (text, true_label) in enumerate(zip(X_test, y_test)):
    preds = {
        "Baseline": int(y_pred_baseline[i]),
        "XGBoost": int(y_pred_xgb[i]),
        "BERT": int(y_pred_bert[i]),
    }
    if y_pred_llm is not None:
        preds["LLM"] = int(y_pred_llm[i])
    
    # Count how many models got it wrong
    wrong_count = sum(1 for p in preds.values() if p != true_label)
    
    if wrong_count > 0:
        errors.append({
            "index": i,
            "text": text[:300] + "..." if len(text) > 300 else text,
            "true_label": int(true_label),
            "true_class": CLASS_NAMES[true_label],
            "predictions": preds,
            "wrong_count": wrong_count
        })

# Sort by how many models got it wrong (most confusing cases first)
errors_sorted = sorted(errors, key=lambda x: -x["wrong_count"])

print(f"Total misclassified examples: {len(errors)} / {len(y_test)}")
print(f"\nTop 10 most confusing cases (all or most models got wrong):")

In [ ]:
# Display top 10 error cases
error_analysis = []

for j, err in enumerate(errors_sorted[:10]):
    print(f"\n{'='*70}")
    print(f"Case {j+1}: True = {err['true_class']} ({err['true_label']})")
    print(f"Predictions: {err['predictions']}")
    print(f"Wrong models: {err['wrong_count']}")
    print(f"Text: {err['text']}")
    
    error_analysis.append({
        "case": j+1,
        "true_label": err['true_class'],
        "predictions": str(err['predictions']),
        "text_preview": err['text'][:100] + "...",
        "analysis": ""  # To be filled manually
    })

# Save error analysis
df_errors = pd.DataFrame(error_analysis)
df_errors.to_csv(REPORTS_DIR / "error_analysis.csv", index=False)
print(f"\n\nSaved error analysis to {REPORTS_DIR / 'error_analysis.csv'}")

## 10. Save Final Results

In [ ]:
# Save all metrics to JSON
final_results = {
    "task": "3-class toxicity classification",
    "dataset": "Jigsaw Toxic Comment (Wikipedia)",
    "test_sample_size": len(y_test),
    "seed": SEED,
    "class_names": CLASS_NAMES,
    "models": [{
        "name": r["model"],
        "accuracy": float(r["accuracy"]),
        "macro_f1": float(r["macro_f1"]),
        "roc_auc": float(r["roc_auc"]) if r["roc_auc"] else None
    } for r in results]
}

with open(REPORTS_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Saved final results to {REPORTS_DIR / 'final_results.json'}")

In [ ]:
# Final summary
print("\n" + "="*60)
print("                    FINAL SUMMARY")
print("="*60)
print(f"Test sample size: {len(y_test)}")
print(f"Class distribution: 0={sum(y_test==0)}, 1={sum(y_test==1)}, 2={sum(y_test==2)}")
print("\nResults (sorted by Macro-F1):")
for r in sorted(results, key=lambda x: -x['macro_f1']):
    auc_str = f", AUC={r['roc_auc']:.4f}" if r['roc_auc'] else ""
    print(f"  {r['model']:25s} Acc={r['accuracy']:.4f}, F1={r['macro_f1']:.4f}{auc_str}")
print("\nSaved files:")
print(f"  - {REPORTS_DIR / 'model_comparison.png'}")
print(f"  - {REPORTS_DIR / 'confusion_matrices.png'}")
print(f"  - {REPORTS_DIR / 'error_analysis.csv'}")
print(f"  - {REPORTS_DIR / 'final_results.json'}")